# Corporate action checks

Runs the two validation scripts in `scripts/` and gives you a way to look at
whatever they flag.

Two things happen here, in this order, and the order matters:

1. **`test_check.py`** builds synthetic files where the answer is known by
   construction and confirms the diagnostic returns the right verdict on each.
   This tests the checker, not your data. A checker that cannot fail is
   indistinguishable from clean data, which is the failure mode this catches.
2. **`check_corporate_actions.py`** runs the diagnostic against the real files
   in `data/equity/`.

Both are run as subprocesses rather than imported, so what executes here is
exactly what would execute from a terminal or a CI step. The scripts stay the
single source of truth and this notebook cannot drift from them.

In [ ]:
import subprocess
import sys
from pathlib import Path

# notebooks/ -> repo root
REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SCRIPTS = REPO / "scripts"
EQUITY = REPO / "data" / "equity"

def run(script, *args):
    """Run a script and stream its output verbatim. Returns the exit code."""
    cmd = [sys.executable, str(SCRIPTS / script), *map(str, args)]
    print("$", " ".join(Path(c).name if c.endswith(".py") else str(c) for c in cmd[1:]))
    print()
    proc = subprocess.run(cmd, capture_output=True, text=True, cwd=str(REPO))
    print(proc.stdout, end="")
    if proc.stderr:
        print("--- stderr ---", file=sys.stderr)
        print(proc.stderr, end="", file=sys.stderr)
    print(f"\nexit status {proc.returncode}")
    return proc.returncode

print("repo    ", REPO)
print("scripts ", SCRIPTS, "exists" if SCRIPTS.is_dir() else "MISSING")
print("equity  ", EQUITY, "exists" if EQUITY.is_dir() else "MISSING")
for folder in ("tanker", "drybulk"):
    files = sorted(p.name for p in (EQUITY / folder).glob("*.csv"))
    print(f"  {folder:9} {len(files):2} files: {', '.join(files)}")

---

## 1. Test the checker

Expect `SELF-TEST PASSED` and exit status **0**.

Inside the test, the diagnostic is expected to exit 1, because two of the four
synthetic cases carry a real defect and a working diagnostic has to fail on
them. Asserting that 1 is part of the test; the 0 is the test itself passing.

If this does not pass, stop. Nothing the next cell says can be trusted.

In [ ]:
rc_selftest = run("test_check.py")

---

## 2. Check the real files

Nine tests: six reverse splits detected on `Close`, three distributions
detected on the step in `Adj Close / Close` at the ex-date.

Those are different tests because they are different defects. Yahoo's `Close`
is already split-adjusted, so a missing split shows as a raw jump there of
about the split factor. A missing distribution leaves the adjusted and raw
columns moving together, so the ratio stays flat while the price falls.

Exit status 0 means every test passed. Three are expected to fail or come back
inconclusive:

| Series | Event | Why it is at risk |
| --- | --- | --- |
| `2020.OL` | NOK 129.5 special dividend, 2026-04-29 | Manifest reports a cumulative adjustment of 1.2324x, far too small to contain it |
| `DSX` | OceanPal spin-off, 2021-11-29 | Yahoo generally does not adjust for spin-offs |
| `DSX` | Diana Containerships spin-off, 2011-01-19 | Same, smaller |

In [ ]:
rc_check = run("check_corporate_actions.py")

---

## 3. Look at what it flagged

The automated check tells you a condition failed. It does not tell you what
happened. For anything that comes back FAIL, WARN or INCONCLUSIVE, plot the
window and look.

`look()` draws the raw and adjusted series either side of a date, plus the
`Adj Close / Close` ratio underneath. On a correctly adjusted distribution the
ratio steps down at the ex-date. On a missed one it runs flat through it.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

def load(ticker, folder):
    for name in (f"{ticker}.csv", f"{ticker.replace('.', '_')}.csv"):
        path = EQUITY / folder / name
        if path.exists():
            break
    else:
        raise FileNotFoundError(f"{ticker} not found in {folder}/")
    df = pd.read_csv(path)
    df["Date"] = pd.to_datetime(df["Date"], errors="coerce", utc=True).dt.tz_localize(None)
    return df.dropna(subset=["Date"]).sort_values("Date").reset_index(drop=True)

def look(ticker, folder, date, days=20, note=""):
    """Plot prices and the adjustment ratio around a corporate action."""
    df = load(ticker, folder)
    target = pd.Timestamp(date)
    lo, hi = target - pd.Timedelta(days=days), target + pd.Timedelta(days=days)
    win = df[(df["Date"] >= lo) & (df["Date"] <= hi)].copy()
    if win.empty:
        print(f"{ticker}: no sessions within {days} days of {date} "
              f"(file covers {df['Date'].min():%Y-%m-%d} to {df['Date'].max():%Y-%m-%d})")
        return

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6), sharex=True,
                                   gridspec_kw={"height_ratios": [2, 1]})
    ax1.plot(win["Date"], win["Close"], marker="o", ms=3, label="Close")
    if "Adj Close" in win.columns:
        ax1.plot(win["Date"], win["Adj Close"], marker="o", ms=3, label="Adj Close")
    ax1.axvline(target, ls="--", lw=1, color="grey")
    ax1.set_title(f"{ticker}  {date}" + (f"  |  {note}" if note else ""))
    ax1.legend(loc="best", fontsize=8)
    ax1.grid(alpha=0.3)

    if "Adj Close" in win.columns:
        ax2.plot(win["Date"], win["Adj Close"] / win["Close"], marker="o", ms=3,
                 color="tab:green")
        ax2.set_ylabel("Adj / Close")
    ax2.axvline(target, ls="--", lw=1, color="grey")
    ax2.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

    # The numbers behind the picture.
    i = win.index[win["Date"] >= target]
    if len(i) and i[0] > win.index[0]:
        k = i[0]
        prev, curr = df.loc[k - 1], df.loc[k]
        print(f"  {prev['Date']:%Y-%m-%d} Close {prev['Close']:,.4f}  ->  "
              f"{curr['Date']:%Y-%m-%d} Close {curr['Close']:,.4f}   "
              f"ratio {curr['Close'] / prev['Close']:,.4f}")
        if "Adj Close" in df.columns:
            r0 = prev["Adj Close"] / prev["Close"]
            r1 = curr["Adj Close"] / curr["Close"]
            print(f"  Adj/Close {r0:.5f} -> {r1:.5f}   step {(r1 / r0 - 1):+.3%}")

In [ ]:
# The three at-risk series. Add or remove calls to match what the check reported.
look("2020.OL", "drybulk", "2026-04-29", days=25, note="NOK 129.5 special dividend")
look("DSX", "drybulk", "2021-11-29", days=25, note="OceanPal spin-off, 1 per 10")
look("DSX", "drybulk", "2011-01-19", days=25, note="Diana Containerships spin-off")

---

## 4. Cross-check against the vendor's own actions list

`pull_prices.py` passes `actions=False`, which discards Yahoo's dividend and
split series. That series is the authoritative list of what *should* have been
adjusted, so comparing against it catches any corporate action missing from
the hand-maintained list in `check_corporate_actions.py`.

Needs network and yfinance. Any split it reports that is not in the script's
list is marked, and that marker is the point of running this.

In [ ]:
rc_online = run("check_corporate_actions.py", "--online")

---

## 5. Where this leaves you

| Exit status | Meaning | What to do |
| --- | --- | --- |
| 0 | Every test passed | Record it, move to Close & Apply |
| 1 | At least one FAIL, or a file missing | Correct or truncate before the model is built |

A failed **split** is a multiplication applied in a rebuild script, the same
shape as the GOGL correction.

A failed **distribution** is harder, because recovering the right adjustment
needs the spun-off entity's opening value. The honest options are to correct
it where that value is obtainable, or to truncate the series before the event
and say why in the README.

`2020.OL` needs truncating at the Q1 2026 fleet sale regardless of what this
reports, since a company holding no vessels is not a dry bulk constituent.

Whatever you do, the decision goes in the README rather than staying in this
notebook. A correction nobody can see is indistinguishable from data that
never needed one.